In [1]:
import json
import os
import math
import librosa

In [2]:
# Local dataset path
DATASET_PATH = r"genres"

In [3]:
# JSON file will be created in the current project folder
JSON_PATH = "data.json"

In [4]:
SAMPLE_RATE = 22050
TRACK_DURATION = 30  # seconds

SAMPLES_PER_TRACK = SAMPLE_RATE * TRACK_DURATION

In [5]:
print("Dataset Path:", DATASET_PATH)
print("Sample Rate:", SAMPLE_RATE)
print("Track Duration:", TRACK_DURATION, "seconds")
print("Samples per Track:", SAMPLES_PER_TRACK)

Dataset Path: genres
Sample Rate: 22050
Track Duration: 30 seconds
Samples per Track: 661500


In [6]:
def save_mfcc(
    dataset_path,
    json_path,
    num_mfcc=13,
    n_fft=2048,
    hop_length=512,
    num_segments=5
):
    """
    Extract MFCC features from the music dataset
    and save them into a JSON file.
    """

    # Dictionary to store mapping, labels and MFCCs
    data = {
        "mapping": [],
        "labels": [],
        "mfcc": []
    }

    genre = {}

    # Calculate samples per segment
    samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)

    # Calculate expected MFCC vectors per segment
    num_mfcc_vectors_per_segment = math.ceil(
        samples_per_segment / hop_length
    )

    # Loop through genre folders
    for i, (dirpath, dirnames, filenames) in enumerate(
        os.walk(dataset_path)
    ):

        # Process only genre folders
        if dirpath != dataset_path:

            # Get genre name
            semantic_label = os.path.basename(dirpath)

            # Store genre mapping
            data["mapping"].append(semantic_label)
            genre[i] = semantic_label

            print("\nProcessing:", semantic_label)

            # Process audio files
            for f in filenames:

                # Only process audio files
                if not f.lower().endswith(".au"):
                    continue

                # Complete file path
                file_path = os.path.join(dirpath, f)

                # Load audio
                signal, sample_rate = librosa.load(
                    file_path,
                    sr=SAMPLE_RATE
                )

                # Process each segment
                for d in range(num_segments):

                    # Calculate start and finish
                    start = samples_per_segment * d
                    finish = start + samples_per_segment

                    # Extract MFCC
                    mfcc = librosa.feature.mfcc(
                        y=signal[start:finish],
                        sr=sample_rate,
                        n_mfcc=num_mfcc,
                        n_fft=n_fft,
                        hop_length=hop_length
                    )

                    # Transpose MFCC
                    mfcc = mfcc.T

                    # Store only correct-sized MFCC
                    if len(mfcc) == num_mfcc_vectors_per_segment:

                        data["mfcc"].append(mfcc.tolist())

                        # Genre label
                        data["labels"].append(i - 1)

    # Save data to JSON
    with open(json_path, "w") as fp:
        json.dump(data, fp, indent=4)

    print("\nMFCC extraction completed!")
    print("JSON file:", json_path)

    return genre

In [7]:
genre_map = save_mfcc(
    DATASET_PATH,
    JSON_PATH,
    num_mfcc=13,
    n_fft=2048,
    hop_length=512,
    num_segments=10
)


Processing: blues


C:\Users\JITENDRA KUMAR\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing: classical

Processing: country

Processing: disco

Processing: hiphop

Processing: jazz

Processing: metal

Processing: pop

Processing: reggae

Processing: rock

MFCC extraction completed!
JSON file: data.json


In [8]:
print("Genre Mapping:")
print(genre_map)

Genre Mapping:
{1: 'blues', 2: 'classical', 3: 'country', 4: 'disco', 5: 'hiphop', 6: 'jazz', 7: 'metal', 8: 'pop', 9: 'reggae', 10: 'rock'}


In [9]:
import json

with open(JSON_PATH, "r") as fp:
    data = json.load(fp)

print("Number of samples:", len(data["mfcc"]))
print("Number of labels:", len(data["labels"]))
print("Genres:", data["mapping"])

Number of samples: 9996
Number of labels: 9996
Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
